# Exploring ColumnTransforation 
```python 
from sklearn.compose import ColumnTransformer
```

### Import required libarary 

In [14]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

### Small Dataframe for example 

In [15]:
df = pd.DataFrame({
    "age": [25, 30, np.nan, 45],
    "salary": [30000, np.nan, 50000, 70000],
    "city": ["Dhaka", "Chattogram", "Dhaka", np.nan],
    "gender": ["Male", "Female", np.nan, "Female"]
})

In [16]:
print("Original DataFrame:\n")
display(df)

Original DataFrame:



,age,salary,city,gender
0,25.0,30000.0,Dhaka,Male
1,30.0,NaN,Chattogram,Female
2,NaN,50000.0,Dhaka,NaN
3,45.0,70000.0,NaN,Female


### Feature labeling 
> We can label the feature by using
```python 
numerical_=df.select_dtypes(include=np.number)
print(numerical_.columns.tolist())
```
**Example output:**
> ['age','salary']

In [17]:
numeric_features = ["age", "salary"]
categorical_features = ["city", "gender"]

### Add encoder to encode categorical columns 
> See more about encoder [see more](preporcessing.ipynb)

In [18]:

try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
except TypeError:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

'''
Pipeline for numeric columns
    1. first step is impute , which will replace Nane value with median 
    2. Then will apply StandardScaler()

'''

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

'''
Pipeline for categorical columns value 
    1. first step is impute, which will replace Nane value with more freequent value (value_counts())
    2. Then will apply one hot encoder that will split one columns to many columns 
'''

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        encoder
    )
])

### Combine many pipelines inside Columns transformer 


In [19]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

### Apply processor for teransforming dataframe 

In [24]:
transformed_data =preprocessor.fit_transform(df)
feature_names = preprocessor.get_feature_names_out()
print("Feature names :\n")
print(feature_names)
print("Processed value :\n")
print(transformed_data)



Feature names :

['numeric__age' 'numeric__salary' 'categorical__city_Chattogram'
 'categorical__city_Dhaka' 'categorical__gender_Female'
 'categorical__gender_Male']
Processed value :

[[-1.         -1.41421356  0.          1.          0.          1.        ]
 [-0.33333333  0.          1.          0.          1.          0.        ]
 [-0.33333333  0.          0.          1.          1.          0.        ]
 [ 1.66666667  1.41421356  0.          1.          1.          0.        ]]


## Transform to DataFrame

In [34]:
processed_df = pd.DataFrame(
    transformed_data,
    columns=feature_names
)

print('Original DataFrame:\n')
print(df)
print("")
print("\nProcessed DataFrame:")
# I renamed the columns to visualized purpose , It's not recomanded 
processed_df=processed_df.rename(columns={
    'numeric__age':'s_age',
    'numeric__salary':'s_salary',
    'categorical__city_Chattogram':'city_Chattogram',
    'categorical__city_Dhaka':'city_Dhaka',
    'categorical__gender_Female':'female',
    'categorical__gender_Male':'male'
})
print(processed_df)

Original DataFrame:

    age   salary        city  gender
0  25.0  30000.0       Dhaka    Male
1  30.0      NaN  Chattogram  Female
2   NaN  50000.0       Dhaka     NaN
3  45.0  70000.0         NaN  Female


Processed DataFrame:
      s_age  s_salary  city_Chattogram  city_Dhaka  female  male
0 -1.000000 -1.414214              0.0         1.0     0.0   1.0
1 -0.333333  0.000000              1.0         0.0     1.0   0.0
2 -0.333333  0.000000              0.0         1.0     1.0   0.0
3  1.666667  1.414214              0.0         1.0     1.0   0.0
